# Thickness conditioning — does attaching the thickness input make better films, or just better numbers?

This notebook gathers the results of the series. It assumes the reading order [audit](audit.ipynb) → [features](features.ipynb) → [baselines](baselines.ipynb) and carries the terminology those established. No model is retrained in this notebook. The numbers — including the five-seed robustness sweep — are read from the frozen records in [`../reports`](../reports): `vae_experiments.json` and `thickness_experiments.json`, written by `scripts/run_vae.py` and `scripts/run_thickness.py` at pinned seeds. The figures shown are the pipeline's own outputs. Reading a record is a correctness check. Every number quoted in the text is computed from the JSON in a cell below, so the prose and the record cannot drift apart.


## How the curves were measured

A UV–vis spectrometer points light through a film and measures how much of each wavelength is absorbed. The optical density is minus the base-10 logarithm of the transmitted fraction: a film transmitting 1% of the light has an optical density of 2 at that wavelength. Because absorption scales with how much material the light passes through, the archive stores thickness-normalized optical density, `q = A/t` in nm⁻¹, with the thickness `t` in nm measured separately by stylus profilometry. Dividing by thickness makes curves of very different films comparable as *material* properties rather than film properties.

That normalization is also this project's central physics question. Beer-Lambert proportionality says absorbance should scale with thickness — twice the film, twice the absorbance. If it held exactly, `q` would be a pure material property and thickness would carry no optical information. The [features notebook](features.ipynb) found that it does not hold: `q` still moves with thickness, re-encoded as shape. So a real film, for modelling, is the *pair* `(q, t)`, and its physically meaningful absorption is `A = q·t` in absorbance units. Every generated film below is a curve *plus* a thickness, and it is judged as a pair.


## Why thickness conditioning?

Thickness is the one thread the series has circled without ever handing to a model. The audit counted it among the fabrication variables complete enough to condition on. The features notebook showed that dividing it out does not remove its influence. After the archive's normalization, thickness information survives inside the normalized curves as shape. And the baselines notebook's generators emit curves only, so whatever thickness information they use is implicit — carried inside codes and components — and no curve-only generator can say what a film of a *requested* thickness would absorb.

Two reasons make conditioning the natural final question. The practical one: conditional generation is the usable product. A materials scientist does not want a random draw from the archive's thickness distribution; they want the absorption spectrum of a film at a thickness they plan to fabricate. The scientific one: because thickness survives inside `q` as shape, a curve-only model already uses thickness information without saying so. Giving a model the thickness *explicitly* asks whether making that information visible helps realism, or whether the model learns to lean on the input and hide errors that the curve-only scores would have caught. The evaluation below is designed so that the answer cannot hide.

What would it mean for thickness conditioning to work? Not a higher score. A score can rise because the physics improved or because the measurement got more forgiving, and this notebook exists to tell the two apart. **Better films** are generated `(q, t)` pairs that hold up as films: curve and thickness coupled the way measured films are coupled, plausible absorbance after the conversion `A = q·t`, novelty that is calibrated rather than copied or overshooting. **Better numbers** are improvements that live in the evaluation instead — the same physical output scoring higher because a metric is computed in units that forgive its error, or a statistic rising for reasons the definition of `q` alone already produces. The two questions below keep that separation.

## The two questions

1. **Better films?** Does attaching a thickness input — feeding `t` as an extra decoder input, so the model can produce a film *at* a requested thickness — make the generated films better in the physical sense: curves and thicknesses that together resemble measured films?
2. **Better numbers?** When the metrics say yes, is the improvement in the physics or in the measurement? Two pitfalls are demonstrated live: an evaluation space that hides errors, and a coupling statistic that the definition of `q` alone nearly reproduces.

In [1]:
import json
from pathlib import Path

import pandas as pd

REPORTS = Path("../reports")
vae_record = json.loads((REPORTS / "vae_experiments.json").read_text(encoding="utf-8"))
thickness_record = json.loads((REPORTS / "thickness_experiments.json").read_text(encoding="utf-8"))
seed_record = json.loads((REPORTS / "seed_diagnostics.json").read_text(encoding="utf-8"))

## What the trained models achieved

The baselines notebook ended with a question:  
The PCA generator reproduces means and spreads but misses the amplitude skewness. Could a small neural model repair the shape?  
The answer, read from the variational record, is the project's first negative result, and it was the expected one. The audit's scree had shown 99% of the population's variance living in four directions, and that expectation was written down before any training:  
On a population this low-dimensional, a neural model matching the linear baseline is a valid outcome. It must earn its complexity through measurable advantage.

Two words need defining, because everything later refers to them. A model's *codes* are the few numbers it compresses each curve into. The PCA analogue is the component scores the baselines notebook drew its Gaussians along. The variational model's training objective carries a penalty, the KL term, that pulls every film's code distribution towards one shared, thickness-blind Gaussian. Sampling codes from that shared Gaussian is what lets the model generate at all. The price is that the codes carry less film-specific detail. The KL is why a VAE's codes are *not* like an autoencoder's tightly reconstructing codes. That difference looks technical here, and it decides the conditioning verdict later.

In [2]:
anchor = next(e for e in vae_record["variational"] if e["kl_weight"] == 0.00664)
summary = vae_record["robustness"]["summary"]
stage5 = pd.DataFrame(
    {
        "PCA generator": {
            "held-out reconstruction (nm²)": vae_record["pca"]["heldout_reconstruction_nm_squared"],
            "novelty ratio": vae_record["pca"]["memorization"]["median_ratio"],
        },
        "autoencoder": {
            "held-out reconstruction (nm²)": vae_record["autoencoder"]["heldout_reconstruction_nm_squared"],
            "novelty ratio": vae_record["autoencoder"]["memorization"]["median_ratio"],
        },
        "VAE, anchor weight": {
            "held-out reconstruction (nm²)": anchor["heldout_reconstruction_nm_squared"],
            "novelty ratio": anchor["memorization"]["median_ratio"],
        },
        "VAE, mean of 3 seeds": {
            "held-out reconstruction (nm²)": summary["variational_reconstruction_nm_squared"]["mean"],
            "novelty ratio": summary["variational_memorization_median_ratio"]["mean"],
        },
    }
).T
stage5["held-out reconstruction (nm²)"] = stage5["held-out reconstruction (nm²)"].map(
    lambda value: f"{value:.2e}"
)
stage5["novelty ratio"] = stage5["novelty ratio"].round(3)
stage5

,held-out reconstruction (nm²),novelty ratio
PCA generator,6.98e-08,1.675
autoencoder,1.33e-07,1.493
"VAE, anchor weight",1.94e-07,1.027
"VAE, mean of 3 seeds",1.69e-07,1.064


Read the table as a tie on fidelity and a separation on novelty. A reconstruction error of 6.98×10⁻⁸ versus 1.94×10⁻⁷ nm² — a factor of three, both far below any perceptible curve difference — does not make the linear generator the better *model*. It makes it the stronger compressor. The novelty ratio is the discriminating number: the PCA generator's curves sit 1.67 times farther from the training set than genuinely new films do, populating coefficient combinations real films never visit, while the VAE holds 1.07 ± 0.05 across seeds — the calibration target of 1 is met. An equal-fidelity generator with calibrated novelty is the outcome that was expected to be valid, and it is the bar this notebook's comparison uses.

The conditioning comparison below asks whether attaching a thickness input moves these numbers — and, more importantly, what it does to the *pairs*.


## The setup: twins, and why films need pair-level scores

Conditioning is architecturally minimal. Each learned model — the autoencoder and the variational model — exists in two twins trained at the identical recipe: the same widths, the same epochs, the same learning rate, the same seeds. The conditioned twin differs by one thing: its decoder receives the film's standardized thickness as a second input alongside the codes. Conditioning adds an input, not a tuned hyperparameter (a training setting that is optimized by trial rather than learned, such as the learning rate). This is deliberate. Suppose the conditioned twin had also received extra tuning of its hyperparameters, a search over learning rates for instance. The question "does conditioning help?" would then silently become "does the better-tuned model help?" — a different question with a more flattering answer. At identical recipes, any difference is attributable to the thickness input alone.

The evaluation is new in the same way. Curve-only scores — the fidelity and novelty metrics of the baselines notebook — cannot judge a *film*, because a curve alone is not a film. The same curve belongs to different physical films at different thicknesses, and the absorption that matters is `A = q·t`. A generator that emits curves and pairs them with arbitrary thicknesses could pass every curve score while producing physically meaningless films. The thickness comparison therefore adds pair-level scores: does the generated population couple curve to thickness the way measured films do, and do the generated films hold up when converted to absorbance, where each curve's error is weighted by the thickness it belongs to?

Two more pair generators calibrate these new scores, neither of which learns anything. The joint resampling floor draws `(curve, thickness)` pairs directly from the training population — coupling copied, fidelity free, novelty exactly zero, as the baselines notebook showed. And a labelled demonstration pairs PCA curves with independently drawn thicknesses on purpose: same marginals, coupling erased by construction. Between them they mark the two ends of the scale every honest pair generator must land inside.


## A metric can hide an error: per-slice reconstruction in two spaces

The first trap is not a property of any model. It is a property of the units a metric is computed in. Take one held-out film, rebuild it, and measure the per-wavelength mistake of the rebuild. In `q`-units the mistake is `e(λ)`. Converting the rebuild to absorbance multiplies it by the film's thickness, because `A_rec - A = t · (q_rec - q)`. The squared error therefore carries a factor `t²`: the *same physical mistake* on a 1300 nm film looks forty times smaller in `q`-units than the same mistake on a 200 nm film. A metric averaged over films of different thickness, computed only in `q`-space, quietly forgives thick-film errors.

Per-slice errors expose this. The held-out films are grouped into four thickness slices of equal film count, and reconstruction errors are averaged inside each slice, per model, in both spaces. The code cell below prints the two tables. The figure shows the `q`-space version. How to read it: one group of five bars per thickness slice, thinnest films on the left and thickest on the right. Within a group there is one bar per generator, and its height is that generator's mean per-film squared error in nm⁻² inside the slice (shorter is better). Solid bars are the unconditioned models and hatched bars their conditioned twins: purple for the autoencoder family, orange for the variational family.


In [3]:
def slice_row(name):
    entry = thickness_record["reconstruction"][name]
    return {
        "q-space (nm²)": [f"{s['mean_error']:.2e}" for s in entry["slices_q_space"]],
        "absorbance space": [f"{s['mean_error']:.4f}" for s in entry["slices_a_space"]],
    }

slice_table = pd.DataFrame(
    {
        "PCA": slice_row("pca"),
        "autoencoder": slice_row("autoencoder"),
        "VAE": slice_row("variational_kl_0.00664"),
        "conditioned AE": slice_row("conditioned_autoencoder"),
        "conditioned VAE": slice_row("conditioned_variational_kl_0.00664"),
    }
).T
slice_table

,q-space (nm²),absorbance space
PCA,"[1.19e-07, 7.82e-08, 5.47e-08, 3.41e-08]","[0.0077, 0.0125, 0.0128, 0.0208]"
autoencoder,"[2.12e-07, 1.62e-07, 1.26e-07, 4.02e-08]","[0.0143, 0.0266, 0.0296, 0.0222]"
VAE,"[2.70e-07, 2.17e-07, 1.72e-07, 1.27e-07]","[0.0180, 0.0353, 0.0391, 0.0754]"
conditioned AE,"[2.56e-07, 1.50e-07, 1.19e-07, 3.83e-08]","[0.0171, 0.0244, 0.0274, 0.0215]"
conditioned VAE,"[2.98e-07, 4.75e-07, 2.68e-07, 9.03e-08]","[0.0206, 0.0777, 0.0587, 0.0523]"


![Per-slice reconstruction errors in q-space, per model](../figures/thickness_slice_errors.png)

*The pipeline's q-space per-slice figure; all five bars share the 10⁻⁷ nm⁻² scale above. The figure flatters the unconditioned VAE, which is the demonstration.*

Now read the unconditioned VAE's solid orange bars. They *fall* from left to right, from 2.70×10⁻⁷ down to 1.27×10⁻⁷ nm²: in `q`-space, the model's own thickest-film slice is its *best*. Look at the same model in the absorbance row of the table and the story reverses: the same films, the same rebuilds, but its errors *rise* to 0.075 — the thickest slice becomes its *worst*, 3.6 times the PCA generator's 0.021 there. Nothing about the model changed between the two readings; only the units did. Two features of the figure do *not* reverse. The grey PCA bars stay the lowest in every group in both spaces. And the conditioning did not remove the reversal: the conditioned VAE's hatched orange bars show the same pattern as their solid twin (its `q`-space errors fall to 0.9×10⁻⁷ on the thickest slice while its absorbance errors rise to 0.078).

The rule this demonstration installs for everything that follows: no reconstruction claim rests on `q`-space per-slice numbers alone. Where the two spaces disagree, the absorbance reading is the physical one. It is the reading in the units a solar-cell physicist actually cares about.


## Does a generated film couple like a real one? The coupling bracket

The pair-level quantity at the centre of the comparison is the *coupling*: the Pearson correlation `r` — the straight-line-trend number — between a film's integrated attenuation `∫q dλ` and its thickness, across a population of films. Measured films couple strongly: thin films show larger `q` integrals, thick films smaller ones, `r = -0.735` on the held-out days.

But a bare number like -0.735 cannot be read on its own, because part of that association is manufactured by the definition of `q`. At a fixed absorbance integral, `∫q = ∫A/t` falls hyperbolically with thickness (pure division, no physics). Over the training population's measured thickness range, 107.5 to 1970.5 nm, a pure `1/t` trend produces `r ≈ -0.76` on its own. The bracket therefore has four readings, written below:

- the **independent pairing**, PCA curves with independently drawn thicknesses: `r = +0.04` — coupling erased by construction, the zero-point;
- the **joint resampling floor**, which replays real `(curve, thickness)` pairs: `r = -0.691` — coupling copied for free, no learning involved;
- the **arithmetic coupling**, the new reference: resample absorbance integrals and thicknesses independently from the training marginals and convert back through `q = A/t` — `r = -0.692`, the coupling the *definition alone* produces;
- the **measured held-out films** themselves: `r = -0.735`.

The measured value sits essentially at the arithmetic level. That is a finding about the population, not the models: the measured `q`–thickness association is nearly all division. Which means the raw `q`-space coupling is already at the level the division alone produces. A generator fed `t` as an input lands near the measured level almost automatically, and "reproduces most of the measured coupling" cannot, by itself, be read as learned thickness physics. The conditioned models' raw couplings — conditioned VAE -0.627, conditioned autoencoder -0.760, thickness-slice generator -0.561, whose band-local description softens the hyperbola — are read inside this bracket, never on their own. The informative physics has moved to the absorbance side, in the next section.


In [4]:
training_thickness = thickness_record["reference_metrics"]["training_thickness_nm"]
print(
    "training thickness range (nm):",
    training_thickness["min"], "to", training_thickness["max"],
)
pd.Series(thickness_record["coupling"]).round(4)

training thickness range (nm): 107.5 to 1970.5


arithmetic_coupling                  -0.6917
conditioned_autoencoder_codes        -0.7604
conditioned_variational_kl_0.00664   -0.6267
heldout_measured                     -0.7350
independent_marginal_pca              0.0377
joint_floor                          -0.6914
thickness_slices                     -0.5605
dtype: float64

![Integrated q against thickness: measured films and generated pairs, with the arithmetic-coupling reference cloud](../figures/thickness_coupling.png)

*Integrated attenuation `∫q` over 300–900 nm against film thickness. One dot per film. Blue: 100 held-out measured films. Grey and green: joint resampling floor and thickness-slice generator. Orange and cyan: conditioned VAE and autoencoder. Red crosses: independent pairing, coupling erased by construction. Purple: the arithmetic coupling — 20,000 draws, hence the densest cloud. Every population falls hyperbolically from left to right, because `q = A/t` builds the fall into any conversion. The measured films trace where real films sit, and the conditioned models land between them and the arithmetic reference. The printed couplings are this picture in numbers.*


## The absorbance side: how absorbance actually scales

If the `q`-space coupling is mostly arithmetic, the physical question lives one conversion away: how does integrated absorbance `∫A dλ` scale with thickness? Beer-Lambert predicts a power law with exponent 1 — twice the film, twice the absorbance. The fitted exponent on the training population is `β̂ = 0.33`, with a bootstrap 95% band of [0.28, 0.39] — decisively below 1. Absorbance grows with thickness, but sublinearly: doubling the film raises the absorbance by roughly 25%.

Why would a real film depart from proportionality? Three mechanisms produce sublinearity, and this number cannot separate them.
- Optical opacity: In the strongly absorbing band a thick film transmits almost nothing. At the plateau height of roughly 10⁻² nm⁻¹, a 1970 nm film would reach an absorbance near 20, far beyond the range a routine spectrophotometer measures. Stray light clamps the reading, and the measured absorbance of the thickest films saturates for an instrumental reason before any material physics is tested.
- Constant offsets: Reflections at the film and substrate interfaces, and a scattering floor, add a thickness-independent contribution. A linear law with an offset lands below exponent 1 in a log–log fit over a finite range.
- Morphology: Thickness correlates with processing conditions that change the material itself — the features notebook's substrate-stack effect is the same systematic effect entering through a different variable.

The source paper reports the Shimadzu 1800 measurement and defines `OD = A/t` through Beer-Lambert without discussing these limits, so the curves alone cannot say how much of the sublinearity is measurement and how much is material. The decomposition needs wavelength-resolved analysis: an instrumental clamp would concentrate the sublinearity in the strongly absorbing band and leave the transparent band linear, while a material cause would act across the spectrum. Band-resolved log–log slopes are therefore an inviting next step, recorded here as a limitation.

One more caution belongs with this number. It is a training-population statistic, and the campaign was not uniform in time. Films coated on the same day share ink, ambient and instrument. Whole days were moved to the held-out side for exactly that reason. The absorbance–thickness association is visibly campaign-dependent (markedly weaker on held-out days than the training value of +0.60 printed below), so population-level absorbance claims carry a date-block caveat, stated wherever they appear.


In [5]:
pd.Series(
    {
        key: (round(value, 4) if isinstance(value, float) else value)
        for key, value in thickness_record["reference_metrics"]["absorbance_scaling"].items()
    }
)

absorbance_coupling_pearson_r                  0.5965
intercept                                      5.0533
population                                   training
slope                                           0.332
slope_ci95                       [0.279843, 0.385832]
dtype: object

## Attach is not help: the conditioned autoencoder, from failure to fix

Start with the mechanics. A conditioned generator makes a film in two steps. It draws a *code* — the short list of numbers that describes a curve's shape, the analogue of the component scores the baselines notebook drew its Gaussians along. It draws a thickness `t` from the measured distribution. The decoder then maps the pair `(code, t)` to a curve, and the generated film is that pair.

Before trusting the thickness input, the check asks how strongly it acts, and where. The *sensitivity* test decodes one fixed set of codes twice: the same codes, first at the 10th percentile of the measured training thicknesses (215 nm), then at the 90th (719 nm). Nothing else changes between the two runs, so any difference between the two curve sets is the thickness input's doing. The test summarizes that difference as the average absolute change per wavelength, in the curves' own unit of nm⁻¹, and sets it against the scale on which real films differ from one another: the held-out population's per-wavelength standard deviation. Both conditioned models respond overwhelmingly. Switching the thickness input from thin to thick moves the average generated curve 289 times that film-to-film spread in the conditioned VAE, and 55 times in the conditioned autoencoder. An input the model ignored would score near zero. The thickness input clearly reaches the generated curves. Reaching them, however, is not the same as making physical films.

An earlier evaluation found the conditioned autoencoder responding at 55 times the population spread while producing *no coupling at all*: `r = -0.035 ± 0.047` across its three seeds, indistinguishable from the independent-pairing demonstration's +0.04. Its generated films were implausible in absorbance space, with a covariance distance of 9.4 against the conditioned VAE's 1.2. In physical terms: the generated films' brightness is unrelated to their stated thickness, as if curve and thickness had been paired at random. That record is the failure, preserved in the git history of `reports/thickness_experiments.json`.

The diagnosis comes down to where the brightness lives. A conditioned decoder sets each generated curve's amplitude through two dials: the codes, and the thickness input. The training objective decides who controls it.

An autoencoder has one objective: reproduce its input curves through the codes alone. Training therefore packs as much film-specific detail as the codes can hold into them, above all the amplitude — the curves' brightness, including their `∫q`. The codes become a brightness dial of their own.

Generation then drew the two dials *independently*: each code from the training codes' Gaussian, each thickness from the measured marginals. Walk through one draw. The code encodes some brightness. The thickness arrives with no relation to it. The decoder combines the two into a curve whose brightness follows the code, while the film claims a thickness the code knows nothing about. Across the whole population, the same mismatch everywhere means the `1/t` trend averages away — the outcome that the labelled independent-pairing demonstration predicts, reproduced by the model's own sampling scheme.

The conditioned VAE never had this problem, for the reason the trained-models section above set up. Its KL term pulls every film's codes towards one thickness-blind prior. The price is that the codes carry less film-specific detail. The reward is that brightness cannot hide in them, so thickness is the only dial left — and the coupling survives.

The fix stays on the sampling side: stop spinning the dials independently. Codes and thickness are drawn *jointly*, from one multivariate Gaussian fitted to the stacked training vectors of codes and standardized log thickness. A Gaussian fitted to raw thickness spans unphysical nonpositive values: the measured range sits only about two standard deviations above zero, roughly one draw in a hundred. Log thickness is positive by construction, and it is the natural coordinate for a positive quantity spanning an order of magnitude. This is the sampling-level version of a principle the floor already embodies: draw the *joint*, never the product of marginals.

The code cell prints the post-fix numbers. The story they tell is failure → diagnosis → fix, with those earlier numbers left standing as the record of the failure. The fixed autoencoder couples at the measured level (-0.760 headline, -0.734 ± 0.023 over five seeds). But its absorbance-space realism stays the weakest of the learned pair generators (covariance distance 2.00 against the conditioned VAE's 1.17 and the floor's 0.29), and its memorization ratio of 1.45 sits well above the VAE's 1.19. Restored coupling is not restored quality.


In [6]:
generation = thickness_record["generation"]
pair_names = ["joint_floor", "independent_marginal_pca", "conditioned_autoencoder_codes", "conditioned_variational_kl_0.00664"]
pair_summary = pd.DataFrame(
    {
        name: {
            "coupling r": round(generation[name]["coupling_pearson_r"], 4),
            "absorbance-space covariance distance": round(
                generation[name]["covariance_vs_heldout_a_space"]["covariance_relative_frobenius"], 4
            ),
            "memorization ratio": round(generation[name]["memorization"]["median_ratio"], 4),
        }
        for name in pair_names
    }
)
robustness = thickness_record["robustness"]["summary"]
formatted = pair_summary.copy()
for column in formatted.columns:
    formatted[column] = formatted[column].map(lambda value: f"{value:.4f}")
formatted.loc["coupling r, over seeds (mean ± std)"] = [
    "—",
    "—",
    f"{robustness['autoencoder_coupling_pearson_r']['mean']:.4f} ± "
    f"{robustness['autoencoder_coupling_pearson_r']['std']:.4f}",
    f"{robustness['variational_coupling_pearson_r']['mean']:.4f} ± "
    f"{robustness['variational_coupling_pearson_r']['std']:.4f}",
]
formatted

,joint_floor,independent_marginal_pca,conditioned_autoencoder_codes,conditioned_variational_kl_0.00664
coupling r,-0.6914,0.0377,-0.7604,-0.6267
absorbance-space covariance distance,0.2864,14.7870,1.9963,1.1706
memorization ratio,0.0000,1.6749,1.4542,1.1884
"coupling r, over seeds (mean ± std)",—,—,-0.7340 ± 0.0226,-0.6006 ± 0.0485


## The thickness response, decomposed: amplitude or shape?

Coupling and sensitivity say *how much* the thickness input moves curves. Neither says *what kind* of motion it produces. The distinction decides what may be claimed, and Beer-Lambert fixes what the expected kind is: at fixed material, changing thickness should only rescale the absorbance. If the thickness input merely rescales curves, the model supports a scaling claim — "conditioning recovers the thickness scaling". Any thickness-dependent *shape* claim (thicker films not only absorb more but switch off differently) needs a coherent shape signal, and shape claims are checked, not assumed.

The check projects the mean sensitivity — the decoding change between the 10th and 90th percentile thicknesses, averaged over codes — onto the held-out mean-curve direction, the curve-height axis. That axis is doubly justified: pure Beer-Lambert scaling acts along it, and it is the population's dominant mode of variation — the features notebook's component 1, which carries 87% of the variance. The *amplitude fraction* says what share of the thickness response is pure rescaling. Whatever is left, called the *residual*, is tested for wavelength coherence. A coherent residual would be the interesting case: it would mean the thickness input changes *where* a film absorbs, not just *how much* — the kind of signal a thin-film physicist would be interested in. Crucially, the measured population gets the same decomposition as an anchor: the held-out films are split at their median thickness (437 nm), the thin-half mean curve minus the thick-half mean curve is projected identically, and the models read against the population's own thickness variation.


In [7]:
sens = thickness_record["reconstruction"]
measured = thickness_record["reference_metrics"]["measured_sensitivity"]
response = pd.DataFrame(
    {
        "measured thin-half vs thick-half": {
            "amplitude fraction": measured["amplitude_fraction"],
            "residual-wavelength r": measured["residual_wavelength_pearson_r"],
        },
        "conditioned VAE": {
            "amplitude fraction": sens["conditioned_variational_kl_0.00664"]["conditioning_sensitivity"]["amplitude_fraction"],
            "residual-wavelength r": sens["conditioned_variational_kl_0.00664"]["conditioning_sensitivity"]["residual_wavelength_pearson_r"],
        },
        "conditioned AE": {
            "amplitude fraction": sens["conditioned_autoencoder"]["conditioning_sensitivity"]["amplitude_fraction"],
            "residual-wavelength r": sens["conditioned_autoencoder"]["conditioning_sensitivity"]["residual_wavelength_pearson_r"],
        },
    }
)
response.round(3)

,measured thin-half vs thick-half,conditioned VAE,conditioned AE
amplitude fraction,0.955,0.888,0.305
residual-wavelength r,-0.709,0.198,0.257


Start with the measured population. Its own thin-versus-thick difference decomposes into 95.5% pure rescaling, with a small coherent residual (r = -0.71) carrying 4.5% of the response's squared magnitude. No model reproduces that residual. The residual's sign fits the previous section's opacity mechanism. Optically clamped blue-band absorbance weighs the thin-versus-thick difference towards short wavelengths, which is what a negative wavelength correlation expresses. Confirming that attribution needs the band-resolved analysis.

The two models then read against this anchor. The conditioned VAE's response is mostly amplitude (88.8%, residual r = +0.20): it follows the population's own variation direction. The conditioned autoencoder's does not — only 30.5% amplitude, with the rest pointing in directions the population's films do not vary along. That is a third sign of the same underlying problem, after the coupling failure and the absorbance realism: its codes and its thickness input disagree about amplitude. Neither conditioned model shows a coherent residual (both r values are small), so no thickness-dependent-shape claim is made. Richer conditioning schemes, which would feed the thickness deeper into the decoder's internal layers, stay unmotivated at this scale: the plain additive input is sufficient for what conditioning can honestly deliver here.


## Seeds and stability

Everything above that involves training a conditioned model is a statistic over several training runs, so this section explains what a training run is and why several are needed. Training a network means improving it in small steps. Over and over, the model looks at a small random batch of the 417 training films, fits them a little better, and moves on. One complete pass over all training films is an *epoch*; the models here train for 300 epochs. Two things are random in this process: the network's starting weights, and which films land in which batch on each pass. Both are fixed by a single number, the *seed*, written in the configuration file. The same seed therefore produces the same training run exactly; a different seed produces a different, equally valid attempt.

The record's sweep retrained each conditioned model at five seeds. The autoencoder converged — finished all 300 passes — at every one. The variational model converged at three of five. Seed 12 blew up at epoch 283 of 300, seed 32 at epoch 133: in both cases the code distributions the model maintains degenerated into invalid numbers, and the training loop detected it and stopped, rather than letting a broken model reach the samples.

Two out of five here is a count and not a rate. What the count needed was a survivor-bias check: are the three converged seeds representative, or did they earn their place by never failing? The check is possible because the seed fixes the whole run. A rerun truncated to 250 epochs — before seed 12's failure at 283 — reproduces the failed trajectory's first 250 passes exactly, and therefore ends in its genuine pre-divergence state. The numbers printed below make the comparison. At that point the diverged seed sits inside the converged seeds' spread on every metric: coupling -0.53 against -0.64 and -0.54, with its covariance distance the *best* of the three and a flat KL history. The failure was sudden and not the visible end of an early drift: the converged-seed statistics are representative. The reliability statement — three of five seeds converge, statistics over the converged three — is stated without an asterisk.


In [8]:
seeds = thickness_record["robustness"]
seed_entries = pd.DataFrame(
    [
        {
            "seed": entry["seed"],
            "converged": not entry.get("diverged", False),
            "coupling r": entry.get("coupling_pearson_r"),
        }
        for family in ("autoencoder", "variational")
        for entry in seeds[family]
    ]
)
truncated = seed_record["truncated_comparison"]["coupling_pearson_r"]
print("conditioned-VAE seeds, coupling r at 250 epochs (truncated rerun):")
print({seed: round(value, 4) for seed, value in truncated.items()})
print("diverged seeds in the 5-seed sweep:", {
    family: [entry["seed"] for entry in seeds[family] if entry.get("diverged")]
    for family in ("autoencoder", "variational")
})
seed_entries

conditioned-VAE seeds, coupling r at 250 epochs (truncated rerun):
{'12': -0.5293, '2': -0.6382, '22': -0.5411}
diverged seeds in the 5-seed sweep: {'autoencoder': [], 'variational': [12, 32]}


,seed,converged,coupling r
0,1,True,-0.760413
1,11,True,-0.742017
2,21,True,-0.692118
3,31,True,-0.738746
4,41,True,-0.736886
5,2,True,-0.626726
6,12,False,NaN
7,22,True,-0.532620
8,32,False,NaN
9,42,True,-0.642405


## Findings

**So which model is best?** It depends on what the model is for. For faithfully reproducing the archive's film distribution, the PCA generator remains the strongest: best held-out reconstruction in both spaces, from four linear components and no training. For generating *new* films at a requested thickness, the conditioned VAE is the best available: the only generator combining calibrated novelty with measured-level coupling. The conditioned autoencoder, after its joint-sampling fix, couples correctly but stays weakest in realism; and the copying floor was never a candidate, only a calibration. No single model wins on every criterion, and that split is itself one of this project's findings. The six findings below substantiate each claim.

1. **Conditioning does not improve held-out reconstruction.** The conditioned twins reconstruct their held-out films no better than the unconditioned versions: the autoencoder is unchanged, and the variational model is worse (2.84×10⁻⁷ against 1.94×10⁻⁷ nm²). The PCA generator remains the strongest model in either space. Fidelity, in other words, is not what conditioning buys.
2. **The `q`-space metric hid thick-film errors.** In the per-slice `q`-space table, the unconditioned VAE seemed to handle the thickest films best. In absorbance space the same films and the same rebuilds show the opposite ordering: the apparent advantage came from the units, whose `t²` forgives a thick film's error, not from the model. Both spaces are therefore always reported together, and no reconstruction claim rests on `q`-space slices alone.
3. **What conditioning improves is joint realism.** The conditioned VAE's films reproduce most of the measured coupling while keeping calibrated novelty (1.19 ± 0.00). After the joint-sampling fix, the conditioned autoencoder couples at the measured level too (-0.734 ± 0.023). It remains, however, the weakest learned generator in absorbance-space realism and memorization: restored coupling is not restored quality.
4. **The measured coupling is mostly arithmetic.** The measured correlation between `∫q` and thickness is -0.735, and the arithmetic coupling — measured `∫A` values paired with independent thicknesses — gives -0.692. The definition of `q` therefore produces nearly the whole association on its own. The informative physics is the absorbance-side sublinearity, `β̂ = 0.33 [0.28, 0.39]`, a population-level characterisation whose decomposition into measurement limits and material effects stays open, with a date-block caveat.
5. **The thickness response is amplitude.** Against the population's own anchor — the measured thin-versus-thick difference, which is 95.5% pure rescaling — the conditioned VAE's response is 88.8% amplitude, following the population's variation direction. The conditioned autoencoder's response is only 30.5% amplitude, with the rest pointing in directions the population's films do not vary along. Neither conditioned model produces a coherent shape residual, so no thickness-dependent-shape claim is made. The measured residual itself (r = -0.71, about 4.5% of the response's squared magnitude) is unexplained by any model here, and stands as a pointer to future work.
6. **Reliability, stated plainly.** Five of five conditioned-autoencoder seeds converged, and three of five conditioned-variational seeds did. A truncated rerun showed the diverged seed's state before failure to be indistinguishable from the converged seeds. All seed statistics are computed over converged seeds only.
